# The race and the baseline

Here is the official starter agent.
**How fast is it, and where does it lose time?**

That is the whole unit.
Before you write a single line of control code, you need to know the race you are entering, the file you submit, and how to measure a lap time you can trust.

The target for this book is a *reasonable ranking* in the Berkeley ROAR Simulation Racing Series.
The Summer 2026 winner ran 3 laps of CARLA Monza in **319.65 s** with zero collisions, and places were separated by tenths of a second.
The book's acceptance bar is: on CARLA Monza, **every one of at least 5 runs is clean** (no major collision), with a **median ≤ 329 s** (within 3% of the winner).

The starter is the floor everybody begins from.
By the end of this lesson you will have its time on three tracks, a map of where the time goes, an honest error bar, and the exact price of one crash.

## 1. The race rules

The pinned official runner (`ROAR_Competition` at commit `ab19e1b`) defines the race.
`racekit` re-implements those rules from their description:

- **3 laps**, scored on **total simulated time**. The waypoint list is repeated once per lap, so "progress" counts all three laps.
- A synchronous world with a **0.05 s control tick**: each tick runs `await agent.step()`, then advances the world one tick.
- **Progress** is the furthest waypoint reached. It is found inside a **forward window of 15 waypoints**, so you cannot skip ahead.
  The race is finished when progress is within **5 waypoints** of the end of the 3-lap list.
- **A major collision** (collision-sensor impulse above 100) sends the car **back to the race start pose** at rest.
  The world then runs **20 ticks (1.0 s) without calling your agent**, and **progress resets to 0**.
  Because the list covers all three laps, a crash **restarts the race while the clock keeps running**.

The published winner's report says "a single crash costs a full lap".
The pinned code is harsher than that, and this book follows the pinned code.
Either way, the conclusion is the same: **zero collisions is the dominant lever on your ranking.**
A clean slow agent beats a fast agent that crashes once.

`racekit` labels its approximations wherever they matter.
The big one in this unit: CARLA detects a crash with a collision sensor; `racekit` counts **leaving the drivable width** (`|offset| > half-width`) as hitting the wall.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races
from racekit.plots import plot_speed_map, plot_telemetry, plot_track
from racekit.runner import (
    CONTROL_DT,
    FINISH_MARGIN,
    PROGRESS_WINDOW,
    RESPAWN_HOLD_TICKS,
    run_race,
)
from racekit.telemetry import CumulativeProgress, race_meta, read_csv, sector_splits, write_csv
from racekit.tracks import load_track

print("control tick:", CONTROL_DT, "s")
print("progress window:", PROGRESS_WINDOW, "waypoints; finish margin:", FINISH_MARGIN, "waypoints")
print("respawn hold:", RESPAWN_HOLD_TICKS, "ticks =", RESPAWN_HOLD_TICKS * CONTROL_DT, "s")
assert abs(RESPAWN_HOLD_TICKS * CONTROL_DT - 1.0) < 1e-12

## 2. The submission: one file, one class

A submission is **one Python file** defining one class, `RoarCompetitionSolution`, with the official constructor and two coroutines:

- `async def initialize(self)` runs once, before the first tick;
- `async def step(self)` runs once per tick: read the sensors, decide, then `await self.vehicle.apply_action({...})`.

Sensors are read with `get_last_gym_observation()`.
The action is a dictionary with exactly six keys: `throttle` in [0, 1], `steer` in [−1, 1] (**positive steers right**), `brake` in [0, 1], `hand_brake`, `reverse` and `target_gear`.

Here is the starter, `agents/unit-01.py`, in full.
Every unit of this book upgrades this one file.

In [ ]:
STARTER = Path("../../projects/racekit/agents/unit-01.py")
with open(STARTER) as handle:
    print(handle.read())

Read it as a controller, not as code:

- **Steering** is proportional: aim at the waypoint `LOOKAHEAD_WAYPOINTS = 3` ahead, compute the heading error to it (wrapped to [−π, π)), and steer by `-STEER_GAIN / sqrt(v) · error / π`.
  The 1/√v gain softens the steering as speed rises.
- **Speed** is proportional too: throttle `SPEED_GAIN · (TARGET_SPEED − v)`, clipped to [0, 1]; a negative command becomes brake.
- It never looks at the corner coming up. It has no idea that a corner limits speed. It simply holds about 20 m/s everywhere.

The tuning knobs are **module-level constants**.
That matters later: `racekit.write_variant` copies the file with a constant changed, so a "variant" is a real submission you can race and check.

The step loop the runner drives looks like this (one coroutine call per tick, never concurrent):

```python
agent = RoarCompetitionSolution(waypoints, vehicle, camera, location, velocity, rpy, occupancy, collision)
await agent.initialize()
while not finished:
    await agent.step()   # your code: read sensors, apply_action
    world.step(0.05)     # physics advances one tick; progress and crashes are checked
```

### The Python 3.8 target

Notebooks in this book run on a modern Python, but **the official environment is Python 3.8 with numpy < 1.25**.
Your submission file must run there. Habits that break it:

- `match` statements;
- `list[int]`, `dict[str, float]` and `X | Y` in annotations or at runtime (use `typing.List`, `Optional`, `Union`);
- `dict | dict` (use `{**a, **b}`);
- `str.removeprefix`, `functools.cache`, `math.lcm`, `np.trapezoid`.

`from __future__ import annotations` hides annotations from a 3.8 run, so a `list[int]` annotation might *seem* to work; the checker in §8 refuses it anyway.

## 3. The first race, deterministically

`run_race(submission, track, car=...)` loads the file, builds the agent with the official arguments and races it.
It is a plain function: no `await` needed.
The default car is a **kinematic bicycle** model; with `jitter=0.0` it is exactly deterministic.

In [ ]:
result = run_race(STARTER, "oval", car=KinematicCar(jitter=0.0))
print(result)
print("finished:", result.finished, "| collisions:", result.collisions, "| ticks:", result.ticks)
print("progress needed:", result.progress_required, "of", result.progress_total, "waypoints")
assert result.finished and result.collisions == 0
assert abs(result.total_time - 223.50) < 1e-9

In [ ]:
again = run_race(STARTER, "oval", car=KinematicCar(jitter=0.0))
print(again.total_time, "s, again")
assert again.total_time == result.total_time

**223.50 s, every time.**
racekit is repeatable; CARLA is not.
That is a feature for debugging and a trap for measuring: the real race has run-to-run noise, and §6 puts it back.

Every number racekit produces is a **model** number.
The `oval` is about 1.4 km; the real Monza is about 5.8 km.
The ranking you care about is earned on CARLA, and the lab at the end of this unit re-measures there.

## 4. Telemetry on `monza-like`

`monza-like` is a generated track *inspired by* Monza: long straights, the chicanes, the two Lesmo corners, Ascari and a Parabolica-like final corner.
Every race returns **telemetry**: one row per control tick, as a NumPy structured array.

We switch on a small actuator noise here (`jitter=0.05`, seed 0); §6 explains why.
This exact race is also the one the lab's sample recording was made from.

In [ ]:
monza = load_track("monza-like")
race = run_race(STARTER, "monza-like", car=KinematicCar(jitter=0.05), seed=0)
telemetry = race.telemetry
print(monza)
print(race)
print("columns:", telemetry.dtype.names)
print(len(telemetry), "rows; first three:")
print(telemetry[:3])
assert abs(race.total_time - 870.30) < 1e-9

The columns:

| column | meaning |
|---|---|
| `t` | simulated seconds at the end of the tick |
| `x`, `y`, `yaw`, `v` | the car's state after the tick (m, m, rad, m/s) |
| `throttle`, `steer`, `brake` | the action the agent applied (before any jitter) |
| `progress` | the furthest waypoint reached, **cumulative over the race** (0 … `progress_total`) |
| `offset` | signed distance from the centreline, left positive (m) |
| `held` | 1 on the respawn-hold ticks after a crash (the agent is not called), else 0 |

Columns are NumPy arrays, so queries are one-liners.
For example, *when does the car first reach 10 m/s?*
`np.argmax` on a boolean array returns the index of the first `True`:

In [ ]:
first = np.argmax(telemetry["v"] >= 10.0)
print("first reaches 10 m/s at t =", round(float(telemetry["t"][first]), 2), "s")
assert abs(telemetry["t"][first] - 2.30) < 1e-9

Now look at the race.
`racekit.plots` draws on a matplotlib axes and returns it, so you can add titles, labels and lines with plain `ax` calls.

In [ ]:
fig, (ax_track, ax_map) = plt.subplots(1, 2, figsize=(13, 5.5))
plot_track(monza, ax=ax_track)
ax_track.set_title("monza-like: centreline, edges and start")
plot_speed_map(telemetry, monza, ax=ax_map)
ax_map.set_title("The starter's speed, coloured by v (m/s)")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
plot_telemetry(telemetry, columns=("v",), ax=ax)
ax.axhline(20.0, color="tab:red", linestyle="--", linewidth=1.0, label="TARGET_SPEED")
ax.set_ylabel("v (m/s)")
ax.set_title("Speed trace: monza-like, 3 laps")
ax.legend()
plt.show()

The speed map is almost one colour, and the speed trace is a launch followed by a flat line.
The starter drives the whole circuit at the same speed: through the straights *and* the chicanes.

## 5. Where the time goes

### The 20 m/s budget

At a constant 20 m/s, a lap of `monza-like` costs its length divided by 20.
Compare that with the starter's measured laps and with the podium.

In [ ]:
splits = race.sector_splits
lap_times = [sum(lap) for lap in splits]
print("lap length:", round(monza.length, 1), "m")
print("at 20 m/s:", round(monza.length / 20.0, 1), "s per lap")
print("starter laps:", np.round(lap_times, 2), "s")
print("mean speed:", round(float(np.mean(telemetry["v"])), 2), "m/s")
print("podium: about", round(319.65 / 3, 2), "s per lap on the real Monza")
assert abs(float(np.mean(telemetry["v"])) - 19.71) < 0.005
assert abs(sum(lap_times) - race.total_time) < 1e-9

The starter needs about 290 s per lap; the podium needs about 106.5 s.
The real Monza is about 5.8 km, so the winners average over 50 m/s.
**The starter is not losing time anywhere in particular. It is losing it everywhere, at the same rate, because it is capped at 20 m/s.**

### Sector splits

The race is cut into 3 laps × 3 sectors of equal waypoint count.
A sector's split is the time from crossing its start boundary to crossing its end boundary.
Splits are how you see *where* a change bought (or lost) time.

`racekit.telemetry.sector_splits(telemetry, progress_total, laps=3, per_lap=3)` computes them from just the `t` and `progress` columns, so it works on anything that has those columns, including a CSV recorded on the real simulator.
The runner uses the same function, so the two agree exactly.

In [ ]:
from_telemetry = sector_splits(telemetry, race.progress_total, laps=3, per_lap=3)
print("sector splits (s), one row per lap:")
print(np.round(from_telemetry, 2))
assert abs(np.max(np.abs(from_telemetry - race.sector_splits))) < 1e-9

standing_start = splits[0][0] - np.mean([splits[1][0], splits[2][0]])
others = [splits[lap][sector] for lap in range(3) for sector in range(3) if (lap, sector) != (0, 0)]
print("lap-1 sector-1 penalty (the standing start):", round(standing_start, 2), "s")
print("every other sector lies in", round(min(others), 2), "to", round(max(others), 2), "s")
assert abs(splits[0][0] - 99.80) < 1e-9
assert max(others) - min(others) < 1.01

Two things to read off the table.

1. **The splits are flat**: the eight warm sectors lie within 1 s of each other, although the sectors contain very different corners.
   Flat splits are the fingerprint of a speed cap: every sector is "length ÷ 20 m/s".
2. **Lap 1, sector 1 is about 3.5 s slower.** That is the standing start: the car begins at rest.
   Nothing in the controller caused it, and the splits isolate it to one cell.
   That is exactly what splits are for.

Is the starter at least taking a clever line? Look at how far it ever strays from the centreline:

In [ ]:
max_offset = float(np.max(np.abs(telemetry["offset"])))
print("max |offset|:", round(max_offset, 2), "m; half-width:", float(np.min(monza.half_width)), "to", float(np.max(monza.half_width)), "m")
assert abs(max_offset - 0.61) < 0.005

At most 0.61 m off the centreline on a track about 11–12 m wide.
It is **not** cutting corners; on this car it does not need to, because 20 m/s is slow.
Speed through corners, braking points and the line are the levers of Parts 1 and 2.

### The same splits from a CSV

The real simulator gives you no `RaceResult`, only what your agent records.
racekit's **v1 telemetry CSV** is the shared format: a one-line header with the integer race facts, then the columns.
`race_meta(result)` builds that header's values, `write_csv` writes the file and `read_csv` returns `(telemetry, meta)`.

In [ ]:
workdir = Path(mkdtemp(prefix="roar-unit-01-"))
csv_path = write_csv(telemetry, workdir / "monza-seed0.csv", race_meta(race))
with open(csv_path) as handle:
    print(handle.readline(), handle.readline())

loaded, meta = read_csv(csv_path)
print(meta)
round_trip = sector_splits(loaded, meta["progress_total"], laps=meta["laps"], per_lap=meta["per_lap"])
print("largest difference from the runner's splits:", float(np.max(np.abs(round_trip - race.sector_splits))), "s")
assert abs(np.max(np.abs(round_trip - race.sector_splits))) < 1e-6

## 6. One run lies

On CARLA the same agent, run twice, gives two different times.
The published reports put run-to-run noise at **about 0.3 s**, while places are separated by **tenths**.
A single-run comparison partly ranks noise.

racekit models this with **actuator jitter**: `KinematicCar(jitter=σ)` adds Gaussian noise `N(0, σ)` to the throttle and the steer on every tick (each clipped to its range; brake untouched), drawn from a generator seeded by `run_race(seed=...)`.
σ = 0.05 was chosen so the **absolute** spread of 3-lap times matches the real ~0.3 s.

`repeat_races(submission, track, n, car_factory=...)` runs `n` seeded races (seeds 0 … n−1), building a fresh car for each, and reports the times, the **median** and the **spread**.
With about five runs we use the **range** (max − min) as the spread: it is conservative and needs no distributional assumptions.

In [ ]:
jittered = lambda: KinematicCar(jitter=0.05)

oval_runs = repeat_races(STARTER, "oval", n=5, car_factory=jittered)
print(oval_runs)
print("times:", np.round(oval_runs.times, 2))
print("in ticks:", np.round(oval_runs.times / CONTROL_DT, 6))
assert oval_runs.all_finished and oval_runs.collisions == 0
assert abs(oval_runs.median - 220.75) < 1e-9 and abs(oval_runs.spread - 0.35) < 1e-9

Three things to notice.

- **The spread is real:** 0.35 s on a 220 s race, from the same file. Seed 0 versus seed 3 alone would "show" a 0.35 s improvement that does not exist.
- **Times are quantized.** The race ends on a tick, so every total is a whole number of 0.05 s ticks. A difference smaller than a tick is not a difference.
- **The jittered runs are about 2.8 s *faster* than the deterministic 223.50 s.**
  Noise is not neutral here: on the oval the speed controller asks for a throttle of only a few hundredths, and noise clipped at 0 has a positive mean, so the noisy car gets a little more throttle on average.
  Compare like with like: the same car factory and the same seeds.

The same measurement on `monza-like` (this is the slow one, about 15 s; we do it once):

In [ ]:
monza_runs = repeat_races(STARTER, "monza-like", n=5, car_factory=jittered)
print(monza_runs)
print("times:", np.round(monza_runs.times, 2))
assert monza_runs.all_finished and monza_runs.collisions == 0
assert abs(monza_runs.median - 870.50) < 1e-9 and abs(monza_runs.spread - 0.45) < 1e-9

### The noise-aware rule

How do you decide that agent A beats agent B?
This book's rule, used from now on for **every** accept-or-reject decision:

> A beats B only when `median(B) − median(A)` is **larger than the bigger of the two spreads**.

`noise_aware_better(a_times, b_times)` implements it.
Try it on a case where the answer is known: the starter against *itself*, on a different set of seeds.

In [ ]:
other_seeds = repeat_races(STARTER, "oval", n=5, seeds=[5, 6, 7, 8, 9], car_factory=jittered)
print("seeds 0-4:", oval_runs)
print("seeds 5-9:", other_seeds)
print("seeds 5-9 beat seeds 0-4?", noise_aware_better(other_seeds.times, oval_runs.times))
print("seeds 0-4 beat seeds 5-9?", noise_aware_better(oval_runs.times, other_seeds.times))
assert not noise_aware_better(other_seeds.times, oval_runs.times)
assert not noise_aware_better(oval_runs.times, other_seeds.times)

Seeds 5–9 have a median 0.10 s lower, and a single pair of runs could have shown 0.35 s.
The rule correctly says **neither** wins.
A real improvement has to clear the noise:

In [ ]:
a = [100.0, 100.2, 100.1]
b = [100.5, 100.6, 100.4]
print("median gap:", round(float(np.median(b) - np.median(a)), 2), "| spreads:", round(max(a) - min(a), 2), round(max(b) - min(b), 2))
print("a beats b?", noise_aware_better(a, b), "| b beats a?", noise_aware_better(b, a))
assert noise_aware_better(a, b) and not noise_aware_better(b, a)

## 7. The price of a crash

Now make the rule in §1 concrete.
A **saboteur** is the starter with one change: from `CRASH_AT = 60.0` s it holds full throttle and a steady right steer of 0.25 until it crashes, then drives normally again.
We build it as a real submission file: the starter's source plus a subclass that replaces `RoarCompetitionSolution`.
It lives in a temporary directory, never next to your work.

In [ ]:
SABOTAGE = """

CRASH_AT = 60.0  # seconds: when the sabotage starts


_Starter = RoarCompetitionSolution


class RoarCompetitionSolution(_Starter):
    \"\"\"The starter, sabotaged: from CRASH_AT it steers right at full throttle until it crashes.\"\"\"

    async def initialize(self) -> None:
        await super().initialize()
        self.steps = 0
        self.crashed = False

    async def step(self) -> None:
        impulse = self.collision_sensor.get_last_gym_observation()
        if float(np.linalg.norm(impulse)) > 100.0:  # the first step after a respawn
            self.crashed = True
        await super().step()
        if not self.crashed and self.steps * 0.05 >= CRASH_AT:
            await self.vehicle.apply_action({
                "throttle": 1.0, "steer": 0.25, "brake": 0.0,
                "hand_brake": 0.0, "reverse": 0, "target_gear": 0,
            })
        self.steps += 1
"""

saboteur_path = workdir / "saboteur.py"
with open(STARTER) as source, open(saboteur_path, "w") as target:
    target.write(source.read() + SABOTAGE)

crash = run_race(saboteur_path, "oval", car=KinematicCar(jitter=0.0))
print(crash)
print("respawn_times:", crash.respawn_times)

Read the crash out of the telemetry: the `held` rows, and `progress` before and after the hold.

In [ ]:
crash_tel = crash.telemetry
held_rows = np.nonzero(crash_tel["held"] == 1)[0]
first_held, last_held = held_rows[0], held_rows[-1]
t_crash = crash.respawn_times[0]
print(len(held_rows), "held rows, t =", round(float(crash_tel["t"][first_held]), 2), "to", round(float(crash_tel["t"][last_held]), 2), "s")
print("progress at the crash:", crash_tel["progress"][first_held - 1], "| during the hold:", crash_tel["progress"][first_held], "| after:", crash_tel["progress"][last_held + 1])

predicted = 223.50 + t_crash + 1.0
print(f"clean {223.50:.2f} + crash at {t_crash:.2f} + hold 1.00 = {predicted:.2f} s; measured {crash.total_time:.2f} s")
assert len(crash.respawn_times) == 1 and abs(t_crash - 60.60) < 1e-9
assert len(held_rows) == RESPAWN_HOLD_TICKS
assert crash_tel["progress"][last_held + 1] <= 1
assert abs(crash.total_time - 285.10) < 1e-9 and abs(crash.total_time - predicted) < CONTROL_DT

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
plot_telemetry(crash_tel, columns=("progress",), ax=ax)
ax.axvline(t_crash, color="tab:red", linestyle="--", linewidth=1.0, label="crash")
ax.set_ylabel("progress (waypoints)")
ax.set_title("A crash at 60.6 s: progress returns to 0 and the race starts again")
ax.legend()
plt.show()
print("splits after the restart:")
print(np.round(crash.sector_splits, 2))

**Total ≈ clean time + t_crash + 1.0 s.**
Everything you drove before the crash is thrown away, and the clock never stopped.
The sector splits restart too: they describe the race that actually counted, the one after the respawn.

Notice how the price depends on *when* you crash.
"A crash costs a lap" is a constant (about 74.5 s on the oval).
The pinned rule costs `t_crash + 1.0`: cheaper than a lap early in lap 1, and nearly a whole extra race late in lap 3.
Under either rule, one crash wipes out every tenth you will ever win by tuning.

## 8. Checking a submission

`check_submission(path)` runs three stages and returns a report with `.ok` and `.findings`:

1. **parse**: the file parses as Python 3.8, and an AST **deny-list** finds no known 3.9+ construct;
2. **shape**: imports are only the standard library, `numpy` and `roar_py_interface`; the class has the official constructor and `async` `initialize`/`step`;
3. **race**: it finishes 3 laps of `test-circuit` with valid actions and `step()` compute p99 ≤ 5 ms.

The deny-list is **not a proof** of 3.8 compatibility. Only running the file on Python 3.8 proves it (the book's isolated 3.8 test does that for every `agents/*.py`).

In [ ]:
report = check_submission(STARTER)
print(report)
assert report.ok

Now break it the way a modern-Python habit would: one helper with a built-in generic annotation.

In [ ]:
modern = workdir / "modern.py"
with open(STARTER) as source, open(modern, "w") as target:
    target.write(source.read() + """

def mean_speed(speeds: list[float]) -> float:
    return sum(speeds) / len(speeds)
""")

bad = check_submission(modern)
print(bad)
assert not bad.ok

The file runs perfectly on this notebook's Python, and it would fail on race day.
Run `check_submission` on every file you are about to submit.

## 9. The bridge: the real simulator

Everything above is a model number.
The lab, `labs/real-sim.ipynb`, runs the same starter on the real stack, on your own machine:

- CARLA 0.9.12 with the Monza map, the pinned `ROAR_Competition` runner, and a Python 3.8 environment;
- `agents/unit-01.py` copied in as `submission.py`;
- a small **recorder** added to `step()` that writes the same v1 CSV you just wrote, one row per step.

The recorder cannot see the runner's progress counter, only its own one-lap waypoint index.
`CumulativeProgress` turns that index into cumulative race progress: it adds a lap at each wrap, and it resets to 0 when the collision sensor reports a major impulse.

In [ ]:
recorder = CumulativeProgress(717)  # the oval has 717 waypoints per lap
quiet = (0.0, 0.0, 0.0)
print([recorder.update(index, quiet) for index in (700, 716, 3, 10)])  # a lap wrap after 716
print(recorder.update(2, (0.0, 1000.0, 0.0)), "after a crash, respawned =", recorder.respawned)

Your recording then goes through exactly the code of §5: `read_csv`, `sector_splits` (with `finished=True`, because a recorder never sees the finishing tick), and the same plots.
Put your CARLA splits next to racekit's, and you will see how far the model is from the real car.

Keep your recordings on your own machine: the book's repository is public.

Next unit: the geometry of a track, and a controller that knows where the corners are.